LoRA with 4-bit quantization to fine-tune a causal LM on a reasoning dataset — only 0.05% of parameters trainable (? standard approach for single-GPU fine-tuning of larger models)

In [1]:
!pip install trl datasets transformers accelerate bitsandbytes peft -q

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

- `nf4` "NormalFloat 4-bit" — a quantization format specifically designed for normally distributed weights like those in transformers
- `padding_side="right"` — pads sequences on the right rather than left. Important for causal LMs where the model reads left to right

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # was float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Model loaded. Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded. Memory footprint: 0.75 GB


- `r=8` is the rank of the adapter matrices. LoRA decomposes a weight update matrix into two smaller matrices of rank r. Lower rank = fewer parameters = faster but less expressive. r=8 is a standard starting point.
- `lora_alpha=16` scales the adapter output by `alpha/r = 16/8 = 2`. Learning rate multiplier for the adapters specifically.
- `target_modules=["q_proj", "v_proj"]` — injects adapters into the query and value projection matrices of every attention layer. These are the most impactful layers to adapt; yweou could also add "k_proj" and "o_proj" for more capacity at the cost of more parameters.



In [4]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


`TinyLlama-Chat` was trained with a specific template using `<|system|>`, `<|user|>`, `<|assistant|>` tags. Using the correct template is important — it's what the model learned to respond to during pretraining. Using the wrong format (like the ### headers from the GPT-2 notebook) is one reason fine-tuning can underperform.

In [5]:
dataset = load_dataset("lighteval/mmlu", "formal_logic", split="test")

def format_example(example):
    choices = ["A", "B", "C", "D"]
    options = "\n".join([f"{choices[i]}: {example['choices'][i]}" for i in range(len(example['choices']))])
    answer = choices[example['answer']]

    # TinyLlama chat format
    text = f"<|system|>You are a helpful assistant that answers multiple choice logic questions.</s><|user|>{example['question']}\n{options}</s><|assistant|>{answer}</s>"
    return {"text": text}

dataset = dataset.map(format_example)
print(dataset[0]["text"])

<|system|>You are a helpful assistant that answers multiple choice logic questions.</s><|user|>Identify the conclusion of the following argument. It is hard not to verify in our peers the same weakened intelligence due to emotions that we observe in our everyday patients. The arrogance of our consciousness, which in general, belongs to the strongest defense mechanisms, blocks the unconscious complexes. Because of this, it is difficult to convince people of the unconscious, and in turn to teach them what their conscious knowledge contradicts. (Sigmund Freud, The Origin and Development of Psychoanalysis)
A: It is hard not to verify in our peers the same weakened intelligence due to emotions that we observe in our everyday patients.
B: The arrogance of our consciousness, which in general, belongs to the strongest defense mechanisms, blocks the unconscious complexes.
C: Because of this, it is difficult to convince people of the unconscious, and in turn to teach them what their conscious kn

## Baseline inference

In [6]:
def ask(question, choices, label=""):
    options = "\n".join([f"{c}: {o}" for c, o in zip(["A","B","C","D"], choices)])
    prompt = f"<|system|>You are a helpful assistant that answers multiple choice logic questions.</s><|user|>{question}\n{options}</s><|assistant|>"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"{label}Model answer: {response.strip()}")

ex = dataset[0]
ask(ex['question'], ex['choices'], label="BEFORE training — ")
print(f"Correct answer: {['A','B','C','D'][ex['answer']]}")

[transformers] Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


BEFORE training — Model answer: Both A and C provide evidence for the
Correct answer: D


In [7]:
sft_config = SFTConfig(
    output_dir="./tinyllama-logic",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    warmup_steps=10,
    max_length=256,
    dataset_text_field="text",
    report_to="none",
    fp16=False,
    bf16=True,
)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)

trainer.train()

/tmp/ipykernel_2728/1769423773.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
5,2.642341
10,2.565136
15,2.245140
20,2.023991


TrainOutput(global_step=24, training_loss=2.294913589954376, metrics={'train_runtime': 240.6081, 'train_samples_per_second': 1.571, 'train_steps_per_second': 0.1, 'total_flos': 463837756022784.0, 'train_loss': 2.294913589954376, 'entropy': 1.8493486568331718, 'mean_token_accuracy': 0.598527517169714, 'num_tokens': 55584.0, 'epoch': 3.0})

In [8]:
ask(ex['question'], ex['choices'], label="AFTER training — ")
print(f"Correct answer: {['A','B','C','D'][ex['answer']]}")

[transformers] Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


AFTER training — Model answer: Option|>
<|>
<|
Correct answer: D


The model learned the response boundary — it started generating at the right position — but 126 examples wasn't enough to consistently output a single letter. The fix would be a larger dataset, more epochs, or a higher rank r in LoRA for more adapter capacity.